<a href="https://colab.research.google.com/github/yy726/tool_use_demo/blob/main/notebooks/01_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Supervised fine-tuning of a tool-calling planner

This notebook fine-tunes **Qwen3-1.7B** (or Qwen3-4B-Instruct) into a *planner*: given a question it calls tools -
`knowledge_search`, `calculator`, `get_weather`, `get_election` - for at most three rounds, several calls per round
when they are independent, and then answers.

What it does, in order:

1. **BFCL** on the base model (general function calling, official `bfcl-eval` pipeline).
2. **PlannerQA-Bench** on the base model (our own held-out planning benchmark, scored on relevance and factuality).
3. Builds the SFT mix: public function-calling data (ToolACE + Hermes) and in-domain demonstrations from a scripted expert.
4. **SFT** with Unsloth + TRL `SFTTrainer`, checkpointing to Google Drive.
5. PlannerQA-Bench again, then pushes the adapter, merged weights and GGUF to the Hugging Face Hub.

`02_grpo.ipynb` continues from the adapter this notebook saves.

**Nothing is lost on a disconnect.** Checkpoints, data, and finished evaluation stages live on Drive: after a
disconnect just *Run all* again. Finished stages are skipped and training resumes from the last checkpoint.

**Before you start:** add `HF_TOKEN` (write access) and `WANDB_API_KEY` to Colab *Secrets* (key icon in the left
bar), and pick an **A100** or **L4** runtime. Run once with `SMOKE = True` (a few minutes, tiny data) to check the
whole pipeline end to end, then set `SMOKE = False`. On a T4 the BFCL stage is skipped: its vLLM server needs a larger GPU.

## 1. Configuration

In [ ]:
SMOKE = True                 # True: tiny end-to-end check (minutes). False: the full run.
MODEL_KEY = "qwen3-1.7b"     # or "qwen3-4b-instruct" (the model of the Unsloth notebooks; ~2.5x slower)
ENABLE_THINKING = False      # Qwen3-1.7B only: let the planner think before calling tools
USE_DRIVE = True             # checkpoints and caches on Google Drive
RESUME = True                # reuse finished stages and checkpoints found on Drive
RUN_BFCL = True              # official BFCL evaluation (A100 / L4; installs vLLM into its own virtualenv)
WANDB_PROJECT = "tool-use-planner"
PUSH_TO_HUB = True           # push adapter + merged model (+ GGUF) to your Hugging Face account
HF_PRIVATE = False
PUSH_GGUF = True             # needed for Unsloth Studio / llama.cpp serving in 03_demo.ipynb
OVERRIDES = {}               # e.g. {"sft_epochs": 1, "n_toolace": 5000}; see planner/config.py

## 2. Install and set up
The install cell follows Unsloth's notebooks and keeps Colab's own torch and numpy. It is silent; the cell after it prints what was installed.

In [ ]:
%%capture
import os
import importlib.metadata as _md
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    def _pin(_p):
        try: return f'{_p}=={_md.version(_p).split("+")[0]}'
        except Exception: return _p
    _pins = " ".join(_pin(_p) for _p in ("numpy", "pillow", "torch", "torchvision"))
    !uv pip install -qqq {_pins} bitsandbytes unsloth
    !uv pip install -qqq triton
    try: _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
    except Exception: _torch_v = ()
    _torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
    !uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==5.15.1 "datasets>=4.7.0,<5.0.0"
!uv pip install --no-deps trl==1.13.0
!uv pip install -qqq --no-deps --force-reinstall "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo" "unsloth @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq bm25s==0.3.16 wandb

In [3]:
import importlib.metadata as _md, sys

def _installed(_p):
    try: return _md.version(_p)
    except Exception: return None

print(" | ".join(f"{_p} {_installed(_p) or 'NOT INSTALLED'}" for _p in ("torch", "numpy", "transformers", "trl", "unsloth", "vllm", "bitsandbytes")))
_missing = [_p for _p in ("torch", "transformers", "trl", "unsloth", "unsloth_zoo", "bm25s") if not _installed(_p)]
assert not _missing, f"The install cell failed for {_missing}. Remove its first line (%%capture), run it again and read the error."
_stale = [_m for _m, _p in (("numpy", "numpy"), ("PIL", "pillow"), ("torch", "torch"))
          if _m in sys.modules and getattr(sys.modules[_m], "__version__", "").split("+")[0] != (_installed(_p) or "").split("+")[0]]
assert not _stale, f"The install replaced {_stale} after this session had loaded it. Choose Runtime > Restart session, then Run all again."

torch 2.14.1+cu130 | numpy 2.4.6 | transformers 5.15.1 | trl 1.13.0 | unsloth 2026.11.4 | vllm NOT INSTALLED | bitsandbytes 0.50.2


In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/yy726/tool_use_demo.git"
REPO_BRANCH = "main"

REPO_DIR = "/content/tool_use_demo"
if os.getcwd() != REPO_DIR and os.path.isdir("planner"):        # run from your own checkout: use it as it is
    REPO_DIR = os.getcwd()
elif os.getcwd() != REPO_DIR and os.path.isdir("../planner"):
    REPO_DIR = os.path.abspath("..")
elif not os.path.isdir(REPO_DIR):
    url = REPO_URL
    try:                                          # private repo: add a GITHUB_TOKEN secret
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token: url = url.replace("https://", f"https://{token}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR], check=True)
else:                                             # a clone from an earlier run: move it to the latest commit of REPO_BRANCH
    for cmd in (["fetch", "--depth", "1", "origin", REPO_BRANCH], ["checkout", "-f", "-B", REPO_BRANCH, "FETCH_HEAD"]):
        done = subprocess.run(["git", "-C", REPO_DIR] + cmd, capture_output=True, text=True)
        if done.returncode: print("WARNING: git", cmd[0], "failed, the code in", REPO_DIR, "may be stale:", done.stderr.strip())
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
for _name in [m for m in sys.modules if m == "planner" or m.startswith("planner.")]:
    del sys.modules[_name]                        # a session that already imported the old code picks up the new code
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "@", head)

In [6]:
import unsloth  # import before transformers / trl so its patches apply
import json, random
from planner.config import get_config
from planner.persist import StageCache, persist_dir, read_secrets
from planner.setup_env import prepare

cfg = get_config("smoke" if SMOKE else "a100", MODEL_KEY, enable_thinking=ENABLE_THINKING, **OVERRIDES)

print("secrets:", read_secrets())                 # HF_TOKEN, WANDB_API_KEY, GITHUB_TOKEN from Colab Secrets
HF_USER = None
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login, whoami
    login(token=os.environ["HF_TOKEN"])
    HF_USER = whoami()["name"]

PERSIST = persist_dir(cfg.exp_name, USE_DRIVE)    # Google Drive: survives a disconnect
LOCAL = f"/content/work/{cfg.exp_name}"            # local disk: large merged weights
os.makedirs(LOCAL, exist_ok=True)
CACHE = StageCache(PERSIST, enabled=RESUME)
CACHE.check_config(cfg.fingerprint())
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())

env = prepare(cfg, os.path.join(PERSIST, "env"))  # world + knowledge base + sft / grpo / eval tasks

secrets: {'HF_TOKEN': True, 'WANDB_API_KEY': True, 'GITHUB_TOKEN': False}
Mounted at /content/drive
persist dir: /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke | finished stages: []
Loaded cached environment from /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/env
Knowledge base: 8794 passages
  sft      96 tasks  {'kb_bridge': 12, 'kb_compare': 12, 'weather_lookup': 12, 'election_lookup': 12, 'fanout_calc': 12, 'tool_bridge': 12, 'composite': 12, 'no_tool': 12}
  grpo    128 tasks  {'kb_bridge': 16, 'kb_compare': 16, 'weather_lookup': 16, 'election_lookup': 16, 'fanout_calc': 16, 'tool_bridge': 16, 'composite': 16, 'no_tool': 16}
  eval     48 tasks  {'kb_bridge': 6, 'kb_compare': 6, 'weather_lookup': 6, 'election_lookup': 6, 'fanout_calc': 6, 'tool_bridge': 6, 'composite': 6, 'no_tool': 6}


## 3. BFCL on the base model

[BFCL](https://gorilla.cs.berkeley.edu/leaderboard.html) measures *general* function calling: unseen function schemas,
parallel calls, refusing to call irrelevant functions, multi-turn tool use. It is evaluation only - nothing is trained
on it. We run the offline categories (about 4,400 test entries) with BFCL's own generation and checker code, and report
the four official groups plus their mean, **BFCL-core**. The agentic group (web search needs a paid key) is not run, so
BFCL-core is not the leaderboard's overall score.

BFCL pins old package versions, so it runs from its own virtualenv and talks to a vLLM server (also in its own virtualenv; the first run installs it, which takes a few minutes). This is why
it runs *before* the training model is loaded. For Qwen3-1.7B the model is evaluated in non-thinking mode at every
stage, so the three checkpoints are compared like for like.

In [7]:
from planner import bfcl

BFCL = {"base": bfcl.run_stage(CACHE, "base", cfg, PERSIST, enabled=RUN_BFCL)}
if BFCL["base"]: print(bfcl.format_table({"base": BFCL["base"]}))

Installing vLLM 0.19.1 into /content/vllm_venv (once per runtime, a few minutes) ...
$ /content/vllm_venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-1.7B --port 1053 --dtype bfloat16 --gpu-memory-utilization 0.85 --trust-remote-code > /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/base/vllm_server.log
$ /content/bfcl_venv/bin/python -m planner.bfcl generate --model planner/qwen3-nothink-FC --test-category simple_python,parallel,irrelevance --backend vllm --skip-server-setup --num-threads 64 --result-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/base/result
$ /content/bfcl_venv/bin/python -m planner.bfcl evaluate --model planner/qwen3-nothink-FC --test-category simple_python,parallel,irrelevance --result-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/base/result --score-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/base/score
| BFCL (official checker) | base |
|---|---|

## 4. Load the model
Unsloth loads the base model in 16-bit and adds a LoRA (rank 32, all projection layers).

In [8]:
from unsloth import FastLanguageModel
from planner.bench import format_report, run_benchmark, text_backend
from planner.chat import setup_chat_format
from planner.tasks import build_prompt
from planner.unsloth_utils import gpu_report, has_vllm, hf_generate_fn, load_lora_weights, load_model, vllm_generate_fn

MAX_NEW = cfg.bench_max_new_tokens * (4 if cfg.enable_thinking else 1)

def bench(name, lora_dir=None):
    """PlannerQA-Bench for the current weights (or a saved LoRA when the vLLM engine is on). Cached."""
    def run():
        if FAST:
            generate = vllm_generate_fn(model, tokenizer, lora_dir, MAX_NEW)
        else:
            FastLanguageModel.for_inference(model)
            generate = hf_generate_fn(model, tokenizer, MAX_NEW, cfg.bench_batch)
        return run_benchmark(env.tasks["eval"], text_backend(fmt.render_prompt, generate), batch_size=cfg.bench_batch, progress=True)
    return CACHE.get_or_run(f"bench_{name}", run)

FAST = False                                       # SFT notebook: plain transformers generation
model, tokenizer = load_model(cfg, fast_inference=False)
gpu_report("after model load")
fmt = setup_chat_format(tokenizer, enable_thinking=cfg.enable_thinking)
print(fmt.render_prompt(build_prompt(env.tasks["eval"][0]))[-700:])

==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 5.15.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
[gpu after model load] total 79.25 GiB | free 75.38 | torch allocated 3.34 | torch reserved 3.35 | used OUTSIDE torch's allocator 0.52
[gpu after model load] processes on this GPU (pid, memory): 436, 3950 MiB
}, "year": {"type": "integer", "description": "The election year, for example 2024."}}, "required": ["region", "year"]}, "return": {"type": "string", "description": "A JSON object with the winner, parties, vote shares, turnout, election date and the region's capital city."}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
People in Yarshire were still arguing about who won the vote. Anyway: Tell me the temperature recorded in Melvale on July 28, 2024.<|im_end|>
<|im_start|>assistant
<think>

</think>




A note on the chat template. TRL's GRPO tool loop needs a template that renders earlier turns identically when tool
results are appended, so it always writes the `<think>...</think>` block of an assistant turn (empty when the model does
not think). `setup_chat_format` uses that same template here, so SFT text, benchmark prompts and GRPO rollouts all show
the model one format.

## 5. PlannerQA-Bench on the base model

800 held-out tasks, 100 per family: `kb_bridge` (multi-hop), `kb_compare` (fan-out), `weather_lookup`,
`election_lookup`, `fanout_calc`, `tool_bridge` (cross-tool multi-hop), `composite` (two-part asks), `no_tool`.
Each rollout gets two scores from 1 to 4 and the final score is their minimum (3 and 4 are "good"):

- **relevance** - did the plan go after every part of the question, without off-topic calls?
- **factuality** - is the answer correct and backed by what the tools returned?

In [9]:
BENCH = {"base": bench("base")}
print(format_report(BENCH))

Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
DEBUG:bm25s:Building index from IDs objects
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  bench: 16/48


Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  bench: 32/48


Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  bench: 48/48
| metric | base |
|---|---|
| good rate | 62.5% |
| relevance (1-4) | 3.33 |
| factuality (1-4) | 2.75 |
| tool cost / query | 0.82 |
| excess cost / query | 0.16 |
| knowledge_search calls / query | 0.62 |
| tool rounds / query | 1.12 |
| invalid call rate | 4.4% |
| final answer rate | 100.0% |
| parallel rate on fan-out tasks | 50.0% |

Good rate by task family:

| family | base |
|---|---|
| kb_bridge | 50.0% |
| kb_compare | 50.0% |
| weather_lookup | 100.0% |
| election_lookup | 83.3% |
| fanout_calc | 33.3% |
| tool_bridge | 50.0% |
| composite | 33.3% |
| no_tool | 100.0% |

Failure tags (share of all tasks):

| tag | base |
|---|---|
| specialized_tool_not_called | 10.4% |
| wrong_tool | 10.4% |
| wrong_arguments | 10.4% |
| not_decomposed | 8.3% |
| query_rewrite_failed | 8.3% |
| multi_hop_incomplete | 10.4% |
| synthesis_error | 4.2% |
| no_tool_called | 2.1% |


## 6. Build the SFT data

| source | role | share |
|---|---|---|
| ToolACE + Hermes function calling | general function calling: arbitrary schemas, parallel calls, declining to call | ~85% |
| scripted expert in the mock environment | our tool set, the three-round protocol, the final `Answer:` line | ~15% |

The expert really runs the tools and only rollouts with a perfect score are kept. It is deliberately plain - it
searches with the question, then with an entity it read about - so GRPO has room to improve on it.

In [10]:
from planner.data_public import load_public_examples, render_examples
from planner.expert import build_expert_traces
from planner.persist import read_jsonl, write_jsonl

SFT_DATA = os.path.join(PERSIST, "sft_train.jsonl")
if RESUME and os.path.exists(SFT_DATA):
    texts = [row["text"] for row in read_jsonl(SFT_DATA)]      # identical data and order after a restart
else:
    FastLanguageModel.for_inference(model)
    plain = hf_generate_fn(model, tokenizer, max_new_tokens=128, batch_size=cfg.bench_batch)
    def direct_answers(queries):                               # no_tool tasks: the base model's own tool-free reply
        return plain([fmt.render_prompt([{"role": "user", "content": q}], tools=None) for q in queries])
    traces = build_expert_traces(env.tasks["sft"], direct_answer_fn=direct_answers)
    public = load_public_examples(cfg.n_toolace, cfg.n_hermes, cfg.seed)
    examples = public + traces
    random.Random(cfg.seed).shuffle(examples)
    texts, dropped = render_examples(examples, fmt, cfg.sft_max_tokens)
    write_jsonl(SFT_DATA, [{"text": t} for t in texts])
    print(f"{len(public)} public + {len(traces)} in-domain examples -> {len(texts)} kept, {dropped} dropped (too long or unrenderable)")
print(len(texts), "training examples")
print(texts[0][-900:])

Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


README.md:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

data.json: reconstructing file:   0%|          |  0.00B / 37.2MB            

data.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11300 [00:00<?, ? examples/s]

Team-ACE/ToolACE: 10337 usable rows (963 dropped), taking 150


README.md:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

func-calling.json: reconstructing file:   0%|          |  0.00B / 20.6MB            

func-calling.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1893 [00:00<?, ? examples/s]

NousResearch/hermes-function-calling-v1: 1090 usable rows (803 dropped), taking 60
210 public + 91 in-domain examples -> 301 kept, 0 dropped (too long or unrenderable)
301 training examples
ing of the searched period in American notation (YYYY-MM-DD)", "type": "string"}, "to": {"description": "End string of the searched period in American notation (YYYY-MM-DD)", "type": "string"}}, "required": ["shareuid", "from", "to"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Could you fetch the historical Net Asset Value for the HDFC Balanced Advantage Fund and the ICICI Prudential Bluechip Fund on the dates September 1, 2026, and November 1, 2026?<|im_end|>
<|im_start|>assistant
<think>

</think>

<tool_call>
{"name": "fetchHistoricalNAV", "arguments": {"string": "01-Sep-2026,01-Nov-2026", "schemeName

## 7. Train

Loss is computed on assistant turns only (`train_on_responses_only`), so tool results and prompts are not learned as
outputs. A checkpoint goes to Drive every `sft_save_steps` steps; if one is already there, training resumes from it.

In [11]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from planner.persist import latest_checkpoint, log_summary, wandb_init
from planner.unsloth_utils import filter_kwargs, save_adapter, tokenizer_kwarg

SFT_DIR = os.path.join(PERSIST, "sft")             # trainer checkpoints
SFT_LORA = os.path.join(PERSIST, "sft_lora")       # final adapter (02_grpo.ipynb starts from this)
os.makedirs(SFT_DIR, exist_ok=True)
wandb_run, REPORT_TO = wandb_init(SFT_DIR, WANDB_PROJECT, f"sft-{cfg.repo_stem}", cfg.to_dict())

if RESUME and os.path.exists(os.path.join(SFT_LORA, "adapter_model.safetensors")):
    load_lora_weights(model, SFT_LORA)
    print("SFT already finished: adapter loaded from Drive")
else:
    FastLanguageModel.for_training(model)
    args = SFTConfig(**filter_kwargs(SFTConfig, dict(
        dataset_text_field = "text",
        max_length = cfg.sft_max_tokens,
        per_device_train_batch_size = cfg.sft_batch,
        gradient_accumulation_steps = cfg.sft_accum,
        num_train_epochs = cfg.sft_epochs,
        max_steps = cfg.sft_max_steps,              # -1: train for num_train_epochs
        learning_rate = cfg.sft_lr,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.03,
        weight_decay = 0.001,
        optim = "adamw_8bit",
        logging_steps = 5,
        seed = cfg.seed,
        output_dir = SFT_DIR,
        save_strategy = "steps",
        save_steps = cfg.sft_save_steps,
        save_total_limit = 2,
        report_to = REPORT_TO,
        run_name = f"sft-{cfg.repo_stem}",
    )))
    trainer = SFTTrainer(model = model, train_dataset = Dataset.from_dict({"text": texts}), args = args, **tokenizer_kwarg(SFTTrainer, tokenizer))
    trainer = train_on_responses_only(trainer, instruction_part = "<|im_start|>user\n", response_part = "<|im_start|>assistant\n")
    checkpoint = latest_checkpoint(SFT_DIR) if RESUME else None
    print("resume from checkpoint:", checkpoint)
    trainer.train(resume_from_checkpoint = checkpoint)
    save_adapter(model, tokenizer, fmt, SFT_LORA)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: darcy-yi-yao (darcy-yi-yao-apple) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


SFTConfig does not accept ['warmup_ratio']: skipped


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/301 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/301 [00:00<?, ? examples/s]

resume from checkpoint: None


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 301 | Num Epochs = 1 | Total steps = 12
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.698089
10,0.265937


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/sft/checkpoint-6/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/sft/checkpoint-12/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/sft_lora/tokenizer_config.json.


## 8. Evaluate the SFT model

In [12]:
BENCH["sft"] = bench("sft")
print(format_report(BENCH))
log_summary(wandb_run, "bench_base", BENCH["base"]["summary"])
log_summary(wandb_run, "bench_sft", BENCH["sft"]["summary"])

Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  bench: 16/48


Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  bench: 32/48


Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  bench: 48/48
| metric | base | sft |
|---|---|---|
| good rate | 62.5% | 72.9% |
| relevance (1-4) | 3.33 | 3.73 |
| factuality (1-4) | 2.75 | 3.31 |
| tool cost / query | 0.82 | 0.82 |
| excess cost / query | 0.16 | 0.04 |
| knowledge_search calls / query | 0.62 | 0.54 |
| tool rounds / query | 1.12 | 1.29 |
| invalid call rate | 4.4% | 1.2% |
| final answer rate | 100.0% | 97.9% |
| parallel rate on fan-out tasks | 50.0% | 88.9% |

Good rate by task family:

| family | base | sft |
|---|---|---|
| kb_bridge | 50.0% | 66.7% |
| kb_compare | 50.0% | 66.7% |
| weather_lookup | 100.0% | 100.0% |
| election_lookup | 83.3% | 100.0% |
| fanout_calc | 33.3% | 83.3% |
| tool_bridge | 50.0% | 100.0% |
| composite | 33.3% | 0.0% |
| no_tool | 100.0% | 66.7% |

Failure tags (share of all tasks):

| tag | base | sft |
|---|---|---|
| specialized_tool_not_called | 10.4% | 0.0% |
| wrong_tool | 10.4% | 2.1% |
| wrong_arguments | 10.4% | 4.2% |
| not_decomposed | 8.3% | 8.3% |
| query_rewrite_fail

## 9. Save and push

Three repositories per checkpoint: the LoRA adapter, the merged 16-bit model (for vLLM, transformers and BFCL) and a
GGUF build (for Unsloth Studio / llama.cpp). The merged weights also stay on local disk for the BFCL run that opens
`02_grpo.ipynb`; after a disconnect that notebook downloads them from the Hub again.

In [13]:
from planner.persist import model_card
from planner.unsloth_utils import export_model

SFT_MERGED = os.path.join(LOCAL, "sft_merged")
card = model_card(
    f"{cfg.repo_stem}-sft",
    cfg.base_model_hf,
    "Tool-calling planner, supervised fine-tuning stage. It plans with `knowledge_search`, `calculator`, `get_weather` and "
    "`get_election` for at most three tool rounds and ends with an `Answer:` line. Code: " + REPO_URL,
    {"PlannerQA-Bench (held-out)": format_report(BENCH), "BFCL": bfcl.format_table({k: v for k, v in BFCL.items() if v}) if BFCL["base"] else "not run"},
)
status = export_model(model, tokenizer, fmt, cfg, "sft", HF_USER, SFT_LORA, SFT_MERGED, card, push = PUSH_TO_HUB, private = HF_PRIVATE, gguf = PUSH_GGUF)
print(status)
if wandb_run: wandb_run.finish()
print("Done. Next: 02_grpo.ipynb (it evaluates this checkpoint on BFCL first).")

Unsloth: Restored added_tokens_decoder metadata in /content/work/planner_qwen3-1.7b_smoke/sft_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/work/planner_qwen3-1.7b_smoke/sft_merged`: 100%|██████████| 1/1 [00:00<00:00, 841.55it/s]


Successfully copied all 1 files from cache to `/content/work/planner_qwen3-1.7b_smoke/sft_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 12.00s/it]


Unsloth: Merge process complete. Saved to `/content/work/planner_qwen3-1.7b_smoke/sft_merged`
[hub] pushed sft adapter
[hub] pushed sft merged model
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


... (10 outputs left out)

Validating                    :           |   0%            

Unsloth: Successfully uploaded GGUF to https://huggingface.co/darcyyao/qwen3-1.7b-planner-smoke-sft-GGUF
Unsloth: Cleaning up temporary files...
[hub] pushed sft GGUF
{'adapter': True, 'merged': True, 'gguf': True}


train/epoch,▁▆█
train/global_step,▁▆█
train/grad_norm,█▁
train/learning_rate,█▁
train/loss,█▁
bench_base/bad_rate,0.375
bench_base/cost,0.82292
bench_base/excess_cost,0.16146
bench_base/factuality,2.75
bench_base/final_answer_rate,1
bench_base/good_rate,0.625


Done. Next: 02_grpo.ipynb (it evaluates this checkpoint on BFCL first).
